# Lezione 6 — Il problema XOR con il machine learning quantistico

**Obiettivi di apprendimento**

- Spiegare perché lo XOR non è linearmente separabile
- Usare l'entanglement e il raggruppamento quantistico (quantum pooling) per risolverlo
- Visualizzare il confine di decisione di un modello quantistico

In [ ]:
!pip install -q cirq tensorflow tensorflow-quantum

## Perché è importante

Il problema XOR (o esclusivo) è celebre nel mondo del machine learning classico: un singolo perceptrone (il neurone più semplice) non riesce a risolverlo, perché i suoi quattro punti non si possono separare con una sola retta. Serve una rete con almeno uno strato nascosto per risolverlo.

In questa lezione verificherai che un circuito quantistico con soli due qubit, grazie all'entanglement, riesce a separare lo XOR senza bisogno di "strati nascosti" classici. È la dimostrazione più chiara che l'entanglement non è solo una curiosità teorica: è una risorsa computazionale reale.

## Perché lo XOR non è linearmente separabile

Lo XOR ha quattro combinazioni di ingresso (0-0, 1-0, 0-1, 1-1) con uscite (-1, +1, +1, -1). Se disegni questi quattro punti su un piano, vedi che non esiste nessuna retta che lasci tutti i "+1" da un lato e tutti i "-1" dall'altro: le due classi sono intrecciate geometricamente, in diagonale. Ti serve una funzione non lineare (una curva, non una retta) per separarle.

## Come l'entanglement fornisce quella "curva"

Il circuito di questa lezione inizia con una porta `CNOT` tra i due qubit di ingresso, che li mette in entanglement prima di applicare qualsiasi rotazione addestrabile. Questo entanglement fa sì che il risultato della misura del circuito non dipenda più da ciascun qubit separatamente, ma dalla relazione tra i due: esattamente la proprietà non lineare che serve per separare lo XOR.

Senza quella `CNOT` iniziale, il circuito si comporta come due classificatori indipendenti e fallisce sullo XOR, proprio come accade a un perceptrone classico.

## Raggruppamento quantistico (pooling): comprimere due qubit nella risposta di uno

Dopo le rotazioni addestrabili, il circuito applica un'operazione di **raggruppamento quantistico** (quantum pooling, la stessa idea del "pooling" di una rete convoluzionale classica, ma per qubit): combina l'informazione dei due qubit in uno solo tramite una sequenza di rotazioni e una `CNOT` finale, in modo che tu debba misurare un solo qubit (il "ricevitore") per ottenere la previsione.

È la stessa tecnica che ricompare nella lezione sull'apprendimento per rinforzo quantistico, per ridurre il numero di qubit da misurare alla fine del circuito.

## Due modi per codificare lo stesso problema

Esistono due varianti comuni per codificare i dati di ingresso nel circuito:

- Codifica a **base computazionale**: ogni bit di ingresso (0 o 1) si traduce direttamente in `cirq.I` (non fare nulla) o `cirq.X` (ribaltare il qubit)
- Codifica a **potenza frazionaria**: ogni ingresso si traduce in `cirq.X(qubit)**ingresso`, il che permette anche di rappresentare valori continui tra zero e uno (non solo 0 e 1 esatti), ed è quella che ti consente di disegnare il confine di decisione completo, non solo i quattro punti discreti dello XOR

Nell'esempio di questa lezione usi la codifica a potenza frazionaria, perché devi esplorare tutto il piano per disegnare il confine di decisione.

## Esempio completo, passo dopo passo

Costruirai il modello in cinque parti: la codifica dei dati, il circuito con la `CNOT` iniziale e le rotazioni addestrabili, il raggruppamento quantistico finale, il modello Keras con il livello `PQC`, e infine l'addestramento con la visualizzazione del confine di decisione.

Inizia dagli import e dalla funzione che codifica un dato classico come rotazione quantistica.

In [ ]:
import tensorflow_quantum as tfq
import tensorflow as tf
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap


def codificar_dato(qubits, circuito, dato):
    """X^i = Rx(pi * i): permite codificar valores continuos entre 0 y 1."""
    circuito.append([cirq.X(qubits[0]) ** dato[0]])
    circuito.append([cirq.X(qubits[1]) ** dato[1]])
    return circuito


def generar_datos_xor(qubits):
    datos = [[0, 0], [1, 0], [0, 1], [1, 1]]
    etiquetas = [-1, 1, 1, -1]
    circuitos = [codificar_dato(qubits, cirq.Circuit(), d) for d in datos]
    return tfq.convert_to_tensor(circuitos), np.array(etiquetas)

Con i dati già definiti, costruisci ora i componenti del circuito addestrabile: la rotazione di un qubit (tre angoli, uno per asse) e il raggruppamento quantistico che combina due qubit in uno solo.

In [ ]:
def rotacion_de_un_qubit(qubit, simbolos):
    return cirq.Circuit(
        cirq.X(qubit) ** simbolos[0],
        cirq.Y(qubit) ** simbolos[1],
        cirq.Z(qubit) ** simbolos[2],
    )


def agrupamiento_dos_qubits(fuente, receptor, simbolos):
    """Combina la información de dos qubits en la respuesta de uno solo (el receptor)."""
    circuito = cirq.Circuit()
    selector_receptor = rotacion_de_un_qubit(receptor, simbolos[0:3])
    selector_fuente = rotacion_de_un_qubit(fuente, simbolos[3:6])
    circuito.append(selector_receptor)
    circuito.append(selector_fuente)
    circuito.append(cirq.CNOT(control=fuente, target=receptor))
    circuito.append(selector_receptor ** -1)
    return circuito

Ora assembla il circuito completo: la `CNOT` iniziale che mette in entanglement i due qubit di ingresso, le sei rotazioni addestrabili (tre per qubit) e il raggruppamento quantistico finale.

In [ ]:
def construir_circuito(qubits):
    x1, y1, z1, x2, y2, z2 = sympy.symbols("x1 y1 z1 x2 y2 z2")
    agrupamiento = sympy.symbols("agrupamiento0:6")
    circuito = cirq.Circuit()
    circuito.append(cirq.CNOT(qubits[0], qubits[1]))  # entrelazamiento clave para el XOR
    circuito.append(cirq.rx(x1).on(qubits[0]))
    circuito.append(cirq.ry(y1).on(qubits[0]))
    circuito.append(cirq.rz(z1).on(qubits[0]))
    circuito.append(cirq.rx(x2).on(qubits[1]))
    circuito.append(cirq.ry(y2).on(qubits[1]))
    circuito.append(cirq.rz(z2).on(qubits[1]))
    circuito += agrupamiento_dos_qubits(qubits[0], qubits[1], agrupamiento)
    return circuito


qubits = [cirq.GridQubit(0, i) for i in range(2)]
print(construir_circuito(qubits))

## Il modello Keras e l'addestramento

Con il circuito già costruito, definisci la metrica di precisione per etichette +1/-1 e monta il modello con `tfq.layers.PQC`, come nella lezione precedente: il livello quantistico riceve i circuiti codificati come ingresso e produce una singola misura (`cirq.Z` sul qubit ricevitore) come uscita.

In [ ]:
def precision_hinge(etiqueta_real, prediccion):
    etiqueta_real = tf.squeeze(etiqueta_real) > 0.0
    prediccion = tf.squeeze(prediccion) > 0.0
    return tf.reduce_mean(tf.cast(etiqueta_real == prediccion, tf.float32))


entrenamiento, etiquetas = generar_datos_xor(qubits)

operador_lectura = [cirq.Z(qubits[1])]
entradas = tf.keras.Input(shape=(), dtype=tf.dtypes.string)
capa_cuantica = tfq.layers.PQC(
    construir_circuito(qubits), operador_lectura, repetitions=32,
    differentiator=tfq.differentiators.ParameterShift(),
    initializer=tf.keras.initializers.Zeros,
)(entradas)
modelo = tf.keras.models.Model(inputs=entradas, outputs=capa_cuantica)
modelo.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
               loss=tf.keras.losses.hinge, metrics=[precision_hinge])

Addestra il modello con le quattro combinazioni dello XOR per 512 epoche. Usi gli stessi dati come validazione, perché lo XOR ha solo quattro punti possibili: non ci sono altri dati da riservare.

In [ ]:
historial = modelo.fit(entrenamiento, etiquetas, epochs=512, batch_size=4,
                        validation_data=(entrenamiento, etiquetas))

## Il confine di decisione

L'addestramento usa solo i quattro punti discreti dello XOR, ma grazie alla codifica a potenza frazionaria puoi valutare il modello in qualsiasi punto continuo del piano [-0.1, 1.1] x [-0.1, 1.1]. Scansiona questa griglia di punti, chiedi una previsione per ciascuno e disegna il risultato con `pcolormesh`.

In [ ]:
mapa_colores = ListedColormap(["midnightblue", "maroon"])
N = 150
x1_malla, x2_malla = np.meshgrid(np.linspace(-0.1, 1.1, N), np.linspace(-0.1, 1.1, N))
puntos = np.c_[x1_malla.ravel(), x2_malla.ravel()]
predicciones = [modelo(tfq.convert_to_tensor([codificar_dato(qubits, cirq.Circuit(), p)])).numpy().flatten()[0]
                for p in puntos]
predicciones = np.asarray(predicciones).reshape(x1_malla.shape)

plt.pcolormesh(x1_malla, x2_malla, predicciones, cmap=mapa_colores)
plt.scatter([0, 1], [0, 1], color="blue", label="-1")
plt.scatter([0, 1], [1, 0], color="red", label="1")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

Nella mappa dei colori finale vedi due regioni diagonali, una per ciascuna classe: esattamente il confine di decisione curvo che un classificatore lineare classico non riesce a disegnare.

## Riepilogo dei punti chiave

- Lo XOR non è linearmente separabile: ti serve un confine di decisione curvo, non una retta
- Una `CNOT` iniziale mette in entanglement i due qubit di ingresso e fornisce la non linearità necessaria per risolvere lo XOR
- Il raggruppamento quantistico (pooling) combina l'informazione di due qubit nella misura di uno solo
- Codificare i dati con potenze frazionarie (`cirq.X(qubit)**dato`) permette di rappresentare valori continui e di disegnare il confine di decisione completo, non solo i quattro punti discreti